## Read data from Volume into Bronze delta table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from delta.tables import DeltaTable


VOLUME_LANDING = "/Volumes/tabular/dataexpert/sm_capstone_stocks/landing/raw/"
BRONZE_TABLE = "tabular.dataexpert.smc_stocks_bronze"
CHECKPOINT = "/Volumes/tabular/dataexpert/sm_capstone_stocks/checkpoints/bronze"

# Drop existing table and checkpoint for a clean start
spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE}")
dbutils.fs.rm(CHECKPOINT, recurse=True)
print("Cleared existing bronze table and checkpoint.")

# Define upsert function
def upsert_to_bronze(batch_df, batch_id):
    batch_df = (
        batch_df
        .withColumn("trade_date", F.to_date(F.col("timestamp")))
        .withColumn("file_source", col("_metadata.file_path"))
        .withColumn("bronze_ingestion_timestamp", F.current_timestamp())
    )

    if spark.catalog.tableExists(BRONZE_TABLE):
        delta_table = DeltaTable.forName(spark, BRONZE_TABLE)
        (
            delta_table.alias("target")
            .merge(
                batch_df.alias("source"),
                """
                target.ticker    = source.ticker AND
                target.timestamp = source.timestamp
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        (
            batch_df.write
            .format("delta")
            .saveAsTable(BRONZE_TABLE)
        )

#Read all the ticker data 
bronze_stream = (spark.readStream
                 .format("cloudFiles")
                 .option("cloudFiles.format", "parquet")
                 .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/schema")
                 .option("cloudFiles.inferSchema", "true")
                 .load(VOLUME_LANDING)
                )

#Writing data to bronze table
(
    bronze_stream.writeStream
            .option("checkpointLocation", CHECKPOINT)
            .trigger(availableNow=True)
            .foreachBatch(upsert_to_bronze)
            .start()
            .awaitTermination()
)

bronze_df = spark.table(BRONZE_TABLE)

print(f"Total rows in bronze table: {bronze_df.count()}")
print("Sample data : ")
display(bronze_df.limit(10))